In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
# load the imdb dataset word index
word_index = imdb.get_word_index()
# reverse the word index to get words from indices
reverse_word_index = {v: k for k, v in word_index.items()}

In [3]:
model = load_model('imdb_rnn_model.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
def decode_review(encoded_review):
    # decode the review text
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

def preproceess_review(review):
    # preprocess the review text
    words = review.lower().split()
    encoded_review = [word_index.get(w, 2) + 3 for w in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review
    

In [5]:
# prediction function
def predict_sentiment(review):
    # preprocess the review
    padded_review = preproceess_review(review)
    # make prediction
    prediction = model.predict(padded_review)
    sentiment = 'positive' if prediction[0][0] > 0.5 else 'negative'
    return sentiment, prediction[0][0]

In [6]:
sample_review = "This movie was fantastic! I loved it."

sentiment, score = predict_sentiment(sample_review)
print(f"Review: {sample_review}")
print(f"Predicted Sentiment: {sentiment}")
print(f"Prediction Score: {score:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 526ms/step
Review: This movie was fantastic! I loved it.
Predicted Sentiment: positive
Prediction Score: 0.9556
